# DeepLabCut Toolbox, labeling with napari
https://github.com/DeepLabCut/DeepLabCut

![alt text](https://images.squarespace-cdn.com/content/v1/57f6d51c9f74566f55ecf271/1d409ffe-c9f4-47e1-bde2-3010c1c40455/naparidlc.png?format=500w)

This notebook demonstrates the necessary steps to use DeepLabCut for your own project.
This shows the most simple code to do so, but many of the functions have additional features, so please check out the overview & the protocol paper!

This notebook illustrates how to:
- create a project
- extract training frames
- label the frames [NEW! with napari]
- plot the labeled images
- create a training set
- train a network
- evaluate a network
- analyze a novel video
- create an automatically labeled video 
- plot the trajectories

This notebook demonstrates the necessary steps to use DeepLabCut for your own project.

This shows the most simple code to do so, but many of the functions have additional features, so please check out the overview & the protocol paper!

Nath\*, Mathis\* et al.: Using DeepLabCut for markerless pose estimation during behavior across species. Nature Protocols, 2019.

Paper: https://www.nature.com/articles/s41596-019-0176-0

Pre-print: https://www.biorxiv.org/content/biorxiv/early/2018/11/24/476531.full.pdf

## Create a new project

It is always good idea to keep the projects separate if you want to use different networks to analze your data. You should use one project if you are tracking similar subjects/items even if in different environments. This function creates a new project with sub-directories and a basic configuration file in the user defined directory otherwise the project is created in the current working directory.

You can always add new videos (for lableing more data) to the project at any stage of the project. 

In [1]:
import deeplabcut

2024-10-21 17:10:00.463335: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-10-21 17:10:00.963891: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-10-21 17:10:02.418553: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


Loading DLC 2.3.4...


In [2]:
task='Reaching' # Enter the name of your experiment Task
experimenter='Mackenzie' # Enter the name of the experimenter
video=['/home/denglab/Shao Ruizinzhu/xinzhu/1/BeA-1-3-me-2023-11-11/videos/yuan72_3_1.avi'
        '/home/denglab/Shao Ruizinzhu/xinzhu/1/BeA-1-3-me-2023-11-11/videos/yuan74_3_1.avi'] # Enter the paths of your videos OR FOLDER you want to grab frames from.

#path_config_file=deeplabcut.create_new_project(task,experimenter,video,copy_videos=True) 

# NOTE: The function returns the path, where your project is. 
# You could also enter this manually (e.g. if the project is already created and you want to pick up, where you stopped...)
path_config_file = '/home/denglab/18T/LinuxDATA/Shao Ruizinzhu/xinzhu2/20241018_PBN_PVH-me-2024-10-21/config.yaml' # Enter the path of the config file that was just created from the above step (check the folder)

## Now, go edit the config.yaml file that was created! 
Add your body part labels, edit the number of frames to extract per video, etc. 

#### Note that you can see more information about ANY function by adding a ? at the end,  i.e. 

In [ ]:
deeplabcut.extract_frames?

## Extract frames from videos 
A key point for a successful feature detector is to select diverse frames, which are typical for the behavior you study that should be labeled.

This function selects N frames either uniformly sampled from a particular video (or folder) ('uniform'). Note: this might not yield diverse frames, if the behavior is sparsely distributed (consider using kmeans), and/or select frames manually etc.

Also make sure to get select data from different (behavioral) sessions and different animals if those vary substantially (to train an invariant feature detector).

Individual images should not be too big (i.e. < 850 x 850 pixel). Although this can be taken care of later as well, it is advisable to crop the frames, to remove unnecessary parts of the frame as much as possible.

Always check the output of cropping. If you are happy with the results proceed to labeling.

In [ ]:
#there are other ways to grab frames, such as uniformly; please see the paper:

#AUTOMATIC:
deeplabcut.extract_frames(path_config_file) 

## Label the extracted frames

Only videos in the config file can be used to extract the frames. Extracted labels for each video are stored in the project directory under the subdirectory **'labeled-data'**. Each subdirectory is named after the name of the video. The toolbox has a labeling toolbox which could be used for labeling. 

In [ ]:
# Attention: If you have not installed the napari-dlc plugin, do so now by running this cell:
!pip install napari-deeplabcut

#if the plugin does not appear upon launch, consider running in the ternimal the above command 
#within the same conda env and then re-starting kernel in your notebook (Kernel > restart).

In [ ]:
# napari will pop up! Please go to plugin > deeplabcut to start:
%gui qt5
import napari
napari.Viewer()

## Check the labels

[OPTIONAL] Checking if the labels were created and stored correctly is beneficial for training, since labeling is one of the most critical parts for creating the training dataset. The DeepLabCut toolbox provides a function `check\_labels'  to do so. It is used as follows:

In [3]:
deeplabcut.check_labels(path_config_file) #this creates a subdirectory with the frames + your labels

Creating images with labels by me.


100%|███████████████████████████████████████████| 20/20 [00:01<00:00, 11.98it/s]

If all the labels are ok, then use the function 'create_training_dataset' to create the training dataset!


If the labels need adjusted, you can use relauch the labeling GUI to move them around, save, and re-plot!

## Create a training dataset

This function generates the training data information for network training based on the pandas dataframes that hold label information. The user can set the fraction of the training set size (from all labeled image in the hd5 file) in the config.yaml file. While creating the dataset, the user can create multiple shuffles if they want to benchmark the performance (typcailly, 1 is what you will set, so you pass nothing!). 

After running this script the training dataset is created and saved in the project directory under the subdirectory **'training-datasets'**

This function also creates new subdirectories under **dlc-models** and appends the project config.yaml file with the correct path to the training and testing pose configuration file. These files hold the parameters for training the network. Such an example file is provided with the toolbox and named as **pose_cfg.yaml**. For most all use cases we have seen, the defaults are perfectly fine.

Now it is the time to start training the network!

In [4]:
deeplabcut.create_training_dataset(path_config_file)
#remember, there are several networks you can pick, the default is resnet-50!

The training dataset is successfully created. Use the function 'train_network' to start training. Happy training!


[(0.95,
  1,
  (array([26, 86,  2, 55, 75, 93, 16, 73, 54, 95, 53, 92, 78, 13,  7, 30, 22,
          24, 33,  8, 43, 62,  3, 71, 45, 48,  6, 99, 82, 76, 60, 80, 90, 68,
          51, 27, 18, 56, 63, 74,  1, 61, 42, 41,  4, 15, 17, 40, 38,  5, 91,
          59,  0, 34, 28, 50, 11, 35, 23, 52, 10, 31, 66, 57, 79, 85, 32, 84,
          14, 89, 19, 29, 49, 97, 98, 69, 20, 94, 72, 77, 25, 37, 81, 46, 39,
          65, 58, 12, 88, 70, 87, 36, 21, 83,  9]),
   array([96, 67, 64, 47, 44])))]

## Start training:

This function trains the network for a specific shuffle of the training dataset. 

In [5]:
deeplabcut.train_network(path_config_file)

Config:
{'all_joints': [[0], [1], [2], [3], [4], [5], [6], [7]],
 'all_joints_names': ['bodypart1',
                      'bodypart2',
                      'bodypart3',
                      'bodypart4',
                      'bodypart5',
                      'bodypart6',
                      'bodypart7',
                      'bodypart8'],
 'alpha_r': 0.02,
 'apply_prob': 0.5,
 'batch_size': 1,
 'contrast': {'clahe': True,
              'claheratio': 0.1,
              'histeq': True,
              'histeqratio': 0.1},
 'convolution': {'edge': False,
                 'emboss': {'alpha': [0.0, 1.0], 'strength': [0.5, 1.5]},
                 'embossratio': 0.1,
                 'sharpen': False,
                 'sharpenratio': 0.3},
 'crop_pad': 0,
 'cropratio': 0.4,
 'dataset': 'training-datasets/iteration-0/UnaugmentedDataSet_20241018_PBN_PVHOct21/20241018_PBN_PVH_me95shuffle1.mat',
 'dataset_type': 'default',
 'decay_steps': 30000,
 'deterministic': False,
 'display_iters': 1000,

Selecting single-animal trainer
Batch Size is 1


/home/denglab/anaconda3/envs/dlc/lib/python3.8/site-packages/tensorflow/python/keras/engine/base_layer_v1.py:1694: UserWarning: `layer.apply` is deprecated and will be removed in a future version. Please use `layer.__call__` method instead.
  warnings.warn('`layer.apply` is deprecated and '
2024-10-21 17:11:18.551993: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
2024-10-21 17:11:18.552041: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1635] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 7621 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3080, pci bus id: 0000:02:00.0, compute capability: 8.6


Loading ImageNet-pretrained resnet_50


2024-10-21 17:11:18.939266: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1635] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 7621 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3080, pci bus id: 0000:02:00.0, compute capability: 8.6
2024-10-21 17:11:19.841644: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:353] MLIR V1 optimization pass is not enabled
2024-10-21 17:11:21.442429: W tensorflow/c/c_api.cc:300] Operation '{name:'Momentum/update_resnet_v1_50/conv1/weights/ResourceApplyMomentum' id:6194 op device:{requested: '', assigned: ''} def:{{{node Momentum/update_resnet_v1_50/conv1/weights/ResourceApplyMomentum}} = ResourceApplyMomentum[T=DT_FLOAT, _class=["loc:@resnet_v1_50/conv1/weights"], _has_manual_control_dependencies=true, use_locking=false, use_nesterov=false](resnet_v1_50/conv1/weights, resnet_v1_50/conv1/weights/Momentum, Placeholder_5, gradients/resnet_v1_50/conv1/Conv2D_grad/tuple/control_dependency_1, Momentum/momentum)}}' was chan

...

iteration: 388000 loss: 0.0019 lr: 0.02
iteration: 389000 loss: 0.0019 lr: 0.02
iteration: 390000 loss: 0.0019 lr: 0.02
iteration: 391000 loss: 0.0019 lr: 0.02
iteration: 392000 loss: 0.0019 lr: 0.02
iteration: 393000 loss: 0.0019 lr: 0.02
iteration: 394000 loss: 0.0020 lr: 0.02
iteration: 395000 loss: 0.0020 lr: 0.02
iteration: 396000 loss: 0.0019 lr: 0.02
iteration: 397000 loss: 0.0019 lr: 0.02
iteration: 398000 loss: 0.0019 lr: 0.02
iteration: 399000 loss: 0.0019 lr: 0.02
iteration: 400000 loss: 0.0019 lr: 0.02
iteration: 401000 loss: 0.0020 lr: 0.02
iteration: 402000 loss: 0.0020 lr: 0.02
iteration: 403000 loss: 0.0019 lr: 0.02
iteration: 404000 loss: 0.0019 lr: 0.02
iteration: 405000 loss: 0.0019 lr: 0.02
iteration: 406000 loss: 0.0019 lr: 0.02
iteration: 407000 loss: 0.0020 lr: 0.02
iteration: 408000 loss: 0.0019 lr: 0.02
iteration: 409000 loss: 0.0019 lr: 0.02
iteration: 410000 loss: 0.0019 lr: 0.02
iteration: 411000 loss: 0.0020 lr: 0.02
iteration: 412000 loss: 0.0019 lr: 0.02


iteration: 589000 loss: 0.0015 lr: 0.002
iteration: 590000 loss: 0.0016 lr: 0.002
iteration: 591000 loss: 0.0015 lr: 0.002
iteration: 592000 loss: 0.0015 lr: 0.002
iteration: 593000 loss: 0.0016 lr: 0.002
iteration: 594000 loss: 0.0016 lr: 0.002
iteration: 595000 loss: 0.0016 lr: 0.002
iteration: 596000 loss: 0.0015 lr: 0.002
iteration: 597000 loss: 0.0015 lr: 0.002
iteration: 598000 loss: 0.0016 lr: 0.002
iteration: 599000 loss: 0.0016 lr: 0.002
iteration: 600000 loss: 0.0015 lr: 0.002
iteration: 601000 loss: 0.0015 lr: 0.002
iteration: 602000 loss: 0.0016 lr: 0.002
iteration: 603000 loss: 0.0016 lr: 0.002
iteration: 604000 loss: 0.0015 lr: 0.002
iteration: 605000 loss: 0.0016 lr: 0.002
iteration: 606000 loss: 0.0015 lr: 0.002
iteration: 607000 loss: 0.0015 lr: 0.002
iteration: 608000 loss: 0.0016 lr: 0.002
iteration: 609000 loss: 0.0015 lr: 0.002
iteration: 610000 loss: 0.0015 lr: 0.002
iteration: 611000 loss: 0.0015 lr: 0.002
iteration: 612000 loss: 0.0015 lr: 0.002
iteration: 61300

iteration: 789000 loss: 0.0015 lr: 0.001
iteration: 790000 loss: 0.0014 lr: 0.001
iteration: 791000 loss: 0.0015 lr: 0.001
iteration: 792000 loss: 0.0015 lr: 0.001
iteration: 793000 loss: 0.0015 lr: 0.001
iteration: 794000 loss: 0.0015 lr: 0.001
iteration: 795000 loss: 0.0015 lr: 0.001
iteration: 796000 loss: 0.0015 lr: 0.001
iteration: 797000 loss: 0.0015 lr: 0.001
iteration: 798000 loss: 0.0015 lr: 0.001
iteration: 799000 loss: 0.0015 lr: 0.001
iteration: 800000 loss: 0.0015 lr: 0.001
iteration: 801000 loss: 0.0015 lr: 0.001
iteration: 802000 loss: 0.0015 lr: 0.001
iteration: 803000 loss: 0.0015 lr: 0.001
iteration: 804000 loss: 0.0015 lr: 0.001
iteration: 805000 loss: 0.0015 lr: 0.001
iteration: 806000 loss: 0.0015 lr: 0.001
iteration: 807000 loss: 0.0015 lr: 0.001
iteration: 808000 loss: 0.0015 lr: 0.001
iteration: 809000 loss: 0.0014 lr: 0.001
iteration: 810000 loss: 0.0016 lr: 0.001
iteration: 811000 loss: 0.0015 lr: 0.001
iteration: 812000 loss: 0.0015 lr: 0.001
iteration: 81300

iteration: 989000 loss: 0.0015 lr: 0.001
iteration: 990000 loss: 0.0015 lr: 0.001
iteration: 991000 loss: 0.0015 lr: 0.001
iteration: 992000 loss: 0.0015 lr: 0.001
iteration: 993000 loss: 0.0015 lr: 0.001
iteration: 994000 loss: 0.0015 lr: 0.001
iteration: 995000 loss: 0.0015 lr: 0.001
iteration: 996000 loss: 0.0015 lr: 0.001
iteration: 997000 loss: 0.0014 lr: 0.001
iteration: 998000 loss: 0.0015 lr: 0.001
iteration: 999000 loss: 0.0014 lr: 0.001
iteration: 1000000 loss: 0.0015 lr: 0.001
iteration: 1001000 loss: 0.0015 lr: 0.001
iteration: 1002000 loss: 0.0015 lr: 0.001
iteration: 1003000 loss: 0.0014 lr: 0.001
iteration: 1004000 loss: 0.0015 lr: 0.001
iteration: 1005000 loss: 0.0015 lr: 0.001
iteration: 1006000 loss: 0.0015 lr: 0.001
iteration: 1007000 loss: 0.0015 lr: 0.001
iteration: 1008000 loss: 0.0015 lr: 0.001
iteration: 1009000 loss: 0.0015 lr: 0.001
iteration: 1010000 loss: 0.0015 lr: 0.001
iteration: 1011000 loss: 0.0015 lr: 0.001
iteration: 1012000 loss: 0.0015 lr: 0.001
ite

The network is now trained and ready to evaluate. Use the function 'evaluate_network' to evaluate the network.


## Start evaluating
This function evaluates a trained model for a specific shuffle/shuffles at a particular state or all the states on the data set (images)
and stores the results as .csv file in a subdirectory under **evaluation-results**

In [ ]:
deeplabcut.evaluate_network(path_config_file, plotting=True)

## Start Analyzing videos
This function analyzes the new video. The user can choose the best model from the evaluation results and specify the correct snapshot index for the variable **snapshotindex** in the **config.yaml** file. Otherwise, by default the most recent snapshot is used to analyse the video.

The results are stored in hd5 file in the same directory where the video resides. 

In [7]:
videofile_path = '/home/denglab/18T/LinuxDATA/Shao Ruizinzhu/xinzhu2/20241018_PBN_PVH-me-2024-10-21/videos'
                 
                  #Enter a folder OR a list of videos to analyze.

deeplabcut.analyze_videos(path_config_file,[videofile_path], videotype='.mp4', save_as_csv = True,cropping = [985,2026,488,1184])

Config:
{'all_joints': [[0], [1], [2], [3], [4], [5], [6], [7]],
 'all_joints_names': ['bodypart1',
                      'bodypart2',
                      'bodypart3',
                      'bodypart4',
                      'bodypart5',
                      'bodypart6',
                      'bodypart7',
                      'bodypart8'],
 'batch_size': 1,
 'crop_pad': 0,
 'dataset': 'training-datasets/iteration-0/UnaugmentedDataSet_20241018_PBN_PVHOct21/20241018_PBN_PVH_me95shuffle1.mat',
 'dataset_type': 'imgaug',
 'deterministic': False,
 'fg_fraction': 0.25,
 'global_scale': 0.8,
 'init_weights': '/home/denglab/anaconda3/envs/dlc/lib/python3.8/site-packages/deeplabcut/pose_estimation_tensorflow/models/pretrained/resnet_v1_50.ckpt',
 'intermediate_supervision': False,
 'intermediate_supervision_layer': 12,
 'location_refinement': True,
 'locref_huber_loss': True,
 'locref_loss_weight': 1.0,
 'locref_stdev': 7.2801,
 'log_dir': 'log',
 'mean_pixel': [123.68, 116.779, 103.939],
 

Overwriting cropping parameters: [985, 2026, 488, 1184]
These are used for all videos, but won't be save to the cfg file.
Using snapshot-1030000 for model /home/denglab/18T/LinuxDATA/Shao Ruizinzhu/xinzhu2/20241018_PBN_PVH-me-2024-10-21/dlc-models/iteration-0/20241018_PBN_PVHOct21-trainset95shuffle1


/home/denglab/anaconda3/envs/dlc/lib/python3.8/site-packages/tensorflow/python/keras/engine/base_layer_v1.py:1694: UserWarning: `layer.apply` is deprecated and will be removed in a future version. Please use `layer.__call__` method instead.
  warnings.warn('`layer.apply` is deprecated and '
2024-10-22 08:52:14.834358: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1635] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 7621 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3080, pci bus id: 0000:02:00.0, compute capability: 8.6


Analyzing all the videos in the directory...
Starting to analyze %  /home/denglab/18T/LinuxDATA/Shao Ruizinzhu/xinzhu2/20241018_PBN_PVH-me-2024-10-21/videos/yuan195.mp4
Loading  /home/denglab/18T/LinuxDATA/Shao Ruizinzhu/xinzhu2/20241018_PBN_PVH-me-2024-10-21/videos/yuan195.mp4
Duration of video [s]:  775.08 , recorded with  19.97 fps!
Overall # of frames:  15480  found with (before cropping) frame dimensions:  3088 2064
Starting to extract posture
Cropping based on the x1 = 985 x2 = 2026 y1 = 488 y2 = 1184. You can adjust the cropping coordinates in the config.yaml file.


100%|█████████████████████████████████████| 15480/15480 [09:00<00:00, 28.65it/s]


...

100%|█████████████████████████████████████| 15839/15839 [09:15<00:00, 28.53it/s]


Saving results in /home/denglab/18T/LinuxDATA/Shao Ruizinzhu/xinzhu2/20241018_PBN_PVH-me-2024-10-21/videos...
Saving csv poses!
Starting to analyze %  /home/denglab/18T/LinuxDATA/Shao Ruizinzhu/xinzhu2/20241018_PBN_PVH-me-2024-10-21/videos/yuan203.mp4
Loading  /home/denglab/18T/LinuxDATA/Shao Ruizinzhu/xinzhu2/20241018_PBN_PVH-me-2024-10-21/videos/yuan203.mp4
Duration of video [s]:  790.4 , recorded with  18.61 fps!
Overall # of frames:  14711  found with (before cropping) frame dimensions:  3088 2064
Starting to extract posture
Cropping based on the x1 = 985 x2 = 2026 y1 = 488 y2 = 1184. You can adjust the cropping coordinates in the config.yaml file.


100%|█████████████████████████████████████| 14711/14711 [09:40<00:00, 25.36it/s]


Saving results in /home/denglab/18T/LinuxDATA/Shao Ruizinzhu/xinzhu2/20241018_PBN_PVH-me-2024-10-21/videos...
Saving csv poses!
The videos are analyzed. Now your research can truly start! 
 You can create labeled videos with 'create_labeled_video'
If the tracking is not satisfactory for some videos, consider expanding the training set. You can use the function 'extract_outlier_frames' to extract a few representative outlier frames.


'DLC_resnet50_20241018_PBN_PVHOct21shuffle1_1030000'

## Extract outlier frames [optional step]

This is an optional step and is used only when the evaluation results are poor i.e. the labels are incorrectly predicted. In such a case, the user can use the following function to extract frames where the labels are incorrectly predicted. This step has many options, so please look at:

In [ ]:
deeplabcut.extract_outlier_frames?

In [ ]:
deeplabcut.extract_outlier_frames(path_config_file,['/videos/video3.avi']) #pass a specific video

## Refine Labels [optional step]
Following the extraction of outlier frames, the user can use the following function to move the predicted labels to the correct location. Thus augmenting the training dataset. 

In [ ]:
#now you can edit the "machine-labeled file" within napari; 
#just again drop the file and images into the workspace after you load the plugin
%gui qt5
import napari
napari.Viewer()

**NOTE:** Afterwards, if you want to look at the adjusted frames, you can load them in the main GUI by running: ``deeplabcut.label_frames(path_config_file)``

(you can add a new "cell" below to add this code!)

#### Once all folders are relabeled, check the labels again! If you are not happy, adjust them in the main GUI:

``deeplabcut.label_frames(path_config_file)``

Check Labels:

``deeplabcut.check_labels(path_config_file)``

In [ ]:
#NOW, merge this with your original data:

deeplabcut.merge_datasets(path_config_file)

## Create a new iteration of training dataset [optional step]
Following the refinement of labels and appending them to the original dataset, this creates a new iteration of training dataset. This is automatically set in the config.yaml file, so let's get training!

In [ ]:
deeplabcut.create_training_dataset(path_config_file)

## Create labeled video
This function is for visualiztion purpose and can be used to create a video in .mp4 format with labels predicted by the network. This video is saved in the same directory where the original video resides. 

THIS HAS MANY FUN OPTIONS! 

``deeplabcut.create_labeled_video(config, videos, videotype='avi', shuffle=1, trainingsetindex=0, filtered=False, save_frames=False, Frames2plot=None, delete=False, displayedbodyparts='all', codec='mp4v', outputframerate=None, destfolder=None, draw_skeleton=False, trailpoints=0, displaycropped=False)``

So please check:

In [9]:
deeplabcut.create_labeled_video?

In [ ]:
videofile_path ='/home/denglab/18T/LinuxDATA/Shao Ruizinzhu/xinzhu2/ASOID-me-2024-03-13/videos2'

deeplabcut.create_labeled_video(path_config_file,videofile_path)

## Plot the trajectories of the analyzed videos
This function plots the trajectories of all the body parts across the entire video. Each body part is identified by a unique color.

In [ ]:
%matplotlib notebook #for making interactive plots.
deeplabcut.plot_trajectories(path_config_file,videofile_path)